# Implied volatility from option prices

Black-Scholes pricing, a robust implied-volatility solver (Newton with a bisection fallback), and a look at the volatility smile and term structure. The option chain here is synthetic, generated from a known smile, so the solver can be checked against the truth.

In [ ]:
import os
from pathlib import Path

# All inputs and outputs live under DATA_DIR: data/ at the repo root by default, whether
# Jupyter was started from the repo root or from notebooks/. Set DATA_DIR to override.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("DATA_DIR", ROOT / "data"))
OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm


def bs_price(S, K, T, r, sigma, call=True):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if call:
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


def bs_vega(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T)


def implied_vol(price, S, K, T, r, call=True, lo=1e-4, hi=5.0, tol=1e-8):
    """Newton steps, falling back to bisection when vega is tiny or a step leaves the bracket."""
    intrinsic = max(S - K * np.exp(-r * T), 0) if call else max(K * np.exp(-r * T) - S, 0)
    if price <= intrinsic + 1e-12:
        return np.nan
    sigma = 0.2
    for _ in range(100):
        diff = bs_price(S, K, T, r, sigma, call) - price
        if abs(diff) < tol:
            return sigma
        if diff > 0:
            hi = sigma
        else:
            lo = sigma
        v = bs_vega(S, K, T, r, sigma)
        step = sigma - diff / v if v > 1e-10 else None
        sigma = step if step is not None and lo < step < hi else 0.5 * (lo + hi)
    return sigma

In [ ]:
rng = np.random.default_rng(SEED)
S, r = 100.0, 0.03
expiries = [30, 60, 90, 180, 365]
strikes = np.arange(70, 131, 5)


def true_smile(K, T):
    m = np.log(K / S)
    return 0.18 + 0.02 / np.sqrt(T) * 0.1 - 0.15 * m + 0.6 * m**2


rows = []
for d in expiries:
    T = d / 365
    for K in strikes:
        call = K >= S  # quote out-of-the-money options
        sig = true_smile(K, T)
        px = bs_price(S, K, T, r, sig, call) * (1 + rng.normal(0, 0.002))
        rows.append(dict(days=d, T=T, strike=K, call=call, price=px, true_vol=sig))
chain = pd.DataFrame(rows)
chain["implied_vol"] = [implied_vol(p, S, k, t, r, c) for p, k, t, c in
                        zip(chain.price, chain.strike, chain["T"], chain.call)]
chain["error_bp"] = (chain.implied_vol - chain.true_vol) * 1e4
chain.head(10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for d, g in chain.groupby("days"):
    ax[0].plot(g.strike, g.implied_vol, marker="o", label=f"{d}d")
ax[0].set_title("Volatility smile by expiry"); ax[0].set_xlabel("strike"); ax[0].legend()
atm = chain.loc[chain.strike == S].sort_values("days")
ax[1].plot(atm.days, atm.implied_vol, marker="o")
ax[1].set_title("At-the-money term structure"); ax[1].set_xlabel("days to expiry")
plt.show()
chain.error_bp.describe()

In [ ]:
# Realised vs implied: compare a rolling realised vol of a simulated path with the ATM implied level.
path = S * np.exp(np.cumsum(rng.normal(0, 0.18 / np.sqrt(252), 500)))
realised = pd.Series(np.log(path)).diff().rolling(21).std() * np.sqrt(252)
realised.plot(figsize=(10, 4), title="21-day realised volatility (simulated)")
plt.axhline(atm.implied_vol.iloc[0], color="red", ls="--", label="30d ATM implied")
plt.legend(); plt.show()